# 01 — Baseline Replication (Saadaoui 2026, JCE)

Python replication of `Saadaoui_JCE_2026.do`, mirroring every figure and table in the paper.

**Inputs:**
- `data/Saadaoui_2026_JCE.dta` (or fallback `original/Saadaoui_2026_JCE.dta`)
- `original/Saadaoui_2026_JCE.log` (for parity checks)

**Outputs:** `figures/`, `results/`

---

## Naming conventions — alignment with Stata `.do` and feature matrix

| Stata `.do` | `.dta` column | This notebook | Notes |
|---|---|---|---|
| `d.llgop` | not stored | `dllgop` | computed as `llgop.diff()` |
| `d.l2lgop` | not stored | `dl2lgop` | computed as `l2lgop.diff()` |
| `d.lpri_jp` etc. | `dlpri_jp` | `dlpri_jp` | **use stored `.dta` column directly** |
| `d2.pri` | `d2pri` | `d2pri` | stored in `.dta` |
| `F2.d2pri` | not stored | `F2_d2pri` | computed as `d2pri.shift(-2)` |

> **Previous version** generated `d_lpri_jp` etc. as computed duplicates and used those in regressions instead of the stored `dlpri_jp`. Results were numerically identical (max diff ≈ 3e-8, float precision only), but the naming was inconsistent with Stata and the feature matrix. Fixed here.

---
## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
import re
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tools.tools import add_constant

HMAX = 48

# ── Paths ──────────────────────────────────────────────────────────────────────
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

ORIGINAL = ROOT / 'original'
FIGURES  = ROOT / 'figures'
RESULTS  = ROOT / 'results'
CACHE    = ROOT / 'data' / 'cache'

dta_data     = ROOT / 'data' / 'Saadaoui_2026_JCE.dta'
dta_original = ORIGINAL / 'Saadaoui_2026_JCE.dta'
DTA = dta_data if dta_data.exists() else dta_original
LOG = ORIGINAL / 'Saadaoui_2026_JCE.log'

for d in [FIGURES, RESULTS, CACHE]:
    d.mkdir(parents=True, exist_ok=True)

print(f'ROOT     → {ROOT}')
print(f'DTA      → {DTA}  (exists: {DTA.exists()})')
print(f'LOG      → {LOG}  (exists: {LOG.exists()})')
print(f'FIGURES  → {FIGURES}')
print(f'RESULTS  → {RESULTS}')

ROOT     → C:\Users\HP\Desktop\replication+contribution
DTA      → C:\Users\HP\Desktop\replication+contribution\data\Saadaoui_2026_JCE.dta  (exists: True)
LOG      → C:\Users\HP\Desktop\replication+contribution\original\Saadaoui_2026_JCE.log  (exists: True)
FIGURES  → C:\Users\HP\Desktop\replication+contribution\figures
RESULTS  → C:\Users\HP\Desktop\replication+contribution\results


---
## Helper functions

In [2]:
def D(s: pd.Series) -> pd.Series:
    """First difference — mirrors Stata `d.varname`."""
    return s.diff()

def F(s: pd.Series, h: int) -> pd.Series:
    """Forward shift by h — mirrors Stata `F{h}.varname`."""
    return s.shift(-h)

def stata_month_to_datetime(period: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(period):
        return pd.to_datetime(period)
    base = pd.Period('1960-01', freq='M')
    numeric = pd.to_numeric(period, errors='coerce')
    return numeric.map(
        lambda m: (base + int(m)).to_timestamp(how='end') if pd.notna(m) else pd.NaT
    )

def set_horizon_axis(ax: plt.Axes) -> None:
    ax.set_xlim(0, HMAX)
    ax.set_xticks(np.arange(0, HMAX + 1, 6))
    ax.set_xlabel('Months')

def set_stata_month_axis(ax: plt.Axes, period_dt: pd.Series) -> None:
    min_year = int(period_dt.dt.year.min())
    max_year = int(period_dt.dt.year.max())
    tick_years = list(range((min_year // 10) * 10, max_year + 1, 10))
    ticks = [pd.Timestamp(year=y, month=1, day=1) for y in tick_years]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{y}m1' for y in tick_years])
    ax.set_xlim(period_dt.min(), period_dt.max())
    ax.set_xlabel('Time')

print('Helpers defined.')

Helpers defined.


---
## Load data

In [3]:
cache_file = CACHE / 'Saadaoui_2026_JCE.parquet'
REFRESH_CACHE = False   # set True to force rebuild from .dta

if cache_file.exists() and not REFRESH_CACHE:
    df = pd.read_parquet(cache_file)
    df = df.sort_values('Period').reset_index(drop=True)
    if 'Period_dt' not in df.columns:
        df['Period_dt'] = stata_month_to_datetime(df['Period'])
    print(f'Loaded from cache: {cache_file}')
else:
    if not DTA.exists():
        raise FileNotFoundError(f'Missing dataset: {DTA}')
    df = pd.read_stata(DTA)
    df = df.sort_values('Period').reset_index(drop=True)
    df['Period_dt'] = stata_month_to_datetime(df['Period'])
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_file, index=False)
    print(f'Loaded from .dta and cached: {cache_file}')

print(f'Shape: {df.shape}')
print(f'Date range: {df["Period_dt"].min().date()} to {df["Period_dt"].max().date()}')

Loaded from cache: C:\Users\HP\Desktop\replication+contribution\data\cache\Saadaoui_2026_JCE.parquet
Shape: (386, 50)
Date range: 1990-01-01 to 2022-02-01


---
## Compute derived columns

`dllgop` and `dl2lgop` are not stored in the `.dta` — Stata computes them inline with `d.llgop` and `d.l2lgop`.  
The `.dta` already contains `dlpri_jp`, `dlpri_aus`, etc. (pre-computed by Saadaoui) — **we use those directly** instead of recomputing them.

In [4]:
# Must-have derived columns
df['dllgop']  = D(df['llgop'])    # Stata: d.llgop
df['dl2lgop'] = D(df['l2lgop'])   # Stata: d.l2lgop
df['F2_d2pri'] = F(df['d2pri'], 2) # Stata: F2.d2pri  (lead test)

# Confirm alliance columns come from .dta (not recomputed)
alliance_cols = [c for c in df.columns if c.startswith('dlpri_')]
print('Alliance controls (from .dta):', sorted(alliance_cols))
print()
print('Derived columns added: dllgop, dl2lgop, F2_d2pri')
print(f'Total columns: {df.shape[1]}')

Alliance controls (from .dta): ['dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_jp', 'dlpri_pak', 'dlpri_rus', 'dlpri_uk', 'dlpri_vn']

Derived columns added: dllgop, dl2lgop, F2_d2pri
Total columns: 53


---
## Control sets

Matching Stata do-file exactly:

| Figure | Stata command | Controls |
|---|---|---|
| Fig 3 (lead test) | `locproj lwti F2_d2pri llwip d.llgop l2lwip d.l2lgop d.lpri_*` | baseline + `dlpri_*` |
| Fig 4 (IV-LP mean) | `locproj lwti lpri llwip d.llgop l2lwip d.l2lgop` | baseline only |
| Fig 5 (quantile) | `locproj lwti lpri llwip dllgop l2lwip dl2lgop` | baseline only |
| Fig C2 (alliances) | `locproj lwti lpri llwip d.llgop l2lwip d.l2lgop dlpri_*` | baseline + `dlpri_*` |

In [5]:
BASE_CONTROLS     = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
ALLIANCE_CONTROLS = [c for c in df.columns if c.startswith('dlpri_')]
LEAD_CONTROLS     = BASE_CONTROLS + ALLIANCE_CONTROLS   # Fig 3
C2_CONTROLS       = BASE_CONTROLS + ALLIANCE_CONTROLS   # Fig C2

print('BASE_CONTROLS    :', BASE_CONTROLS)
print('ALLIANCE_CONTROLS:', sorted(ALLIANCE_CONTROLS))
print(f'LEAD/C2 total    : {len(LEAD_CONTROLS)} controls')

BASE_CONTROLS    : ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
ALLIANCE_CONTROLS: ['dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_jp', 'dlpri_pak', 'dlpri_rus', 'dlpri_uk', 'dlpri_vn']
LEAD/C2 total    : 15 controls


---
## Estimation functions

In [6]:
def add_lagged_controls(
    df: pd.DataFrame,
    y_col: str,
    shock_col: str,
    y_lags: int = 3,
    shock_lags: int = 2,
) -> tuple[pd.DataFrame, list[str]]:
    out = df.copy()
    lag_cols: list[str] = []
    for l in range(1, y_lags + 1):
        c = f'L{l}_{y_col}'
        out[c] = out[y_col].shift(l)
        lag_cols.append(c)
    for l in range(1, shock_lags + 1):
        c = f'L{l}_{shock_col}'
        out[c] = out[shock_col].shift(l)
        lag_cols.append(c)
    return out, lag_cols


def lp_ols(
    df: pd.DataFrame,
    x: str,
    controls: list[str],
    y_col: str = 'lwti',
    y_lags: int = 3,
    shock_lags: int = 2,
    hmax: int = HMAX,
) -> pd.DataFrame:
    """OLS local projections — used for lead test (Figure 3)."""
    work, lag_cols = add_lagged_controls(df, y_col=y_col, shock_col=x,
                                          y_lags=y_lags, shock_lags=shock_lags)
    rhs_cols = [x] + lag_cols + controls
    rows = []
    for h in range(hmax + 1):
        hdf = pd.DataFrame(
            {'y_fwd': F(work[y_col], h), **{c: work[c] for c in rhs_cols}}
        ).replace([np.inf, -np.inf], np.nan).dropna()
        if len(hdf) < 30:
            rows.append((h, np.nan, np.nan))
            continue
        X   = add_constant(hdf[rhs_cols], has_constant='add')
        fit = sm.OLS(hdf['y_fwd'], X).fit(cov_type='HC1')
        rows.append((h, fit.params.get(x, np.nan), fit.bse.get(x, np.nan)))
    irf = pd.DataFrame(rows, columns=['h', 'coef', 'se'])
    irf['lo95'] = irf['coef'] - 1.96 * irf['se']
    irf['hi95'] = irf['coef'] + 1.96 * irf['se']
    return irf


def lp_iv(
    df: pd.DataFrame,
    endog: str,
    instr: str,
    controls: list[str],
    y_col: str = 'lwti',
    y_lags: int = 3,
    shock_lags: int = 2,
    hmax: int = HMAX,
) -> pd.DataFrame:
    """IV-GMM local projections — core estimator (Figures 4, B2, C2)."""
    work, lag_cols = add_lagged_controls(df, y_col=y_col, shock_col=endog,
                                          y_lags=y_lags, shock_lags=shock_lags)
    exog_cols = lag_cols + controls
    rows = []
    for h in range(hmax + 1):
        hdf = pd.DataFrame(
            {
                'y_fwd': F(work[y_col], h),
                endog:   work[endog],
                instr:   work[instr],
                **{c: work[c] for c in exog_cols},
            }
        ).replace([np.inf, -np.inf], np.nan).dropna()
        if len(hdf) < 30:
            rows.append((h, np.nan, np.nan))
            continue
        fit = IV2SLS(
            dependent=hdf['y_fwd'],
            exog=add_constant(hdf[exog_cols], has_constant='add'),
            endog=hdf[endog],
            instruments=hdf[instr],
        ).fit(cov_type='robust', debiased=True)
        rows.append((h, fit.params.get(endog, np.nan), fit.std_errors.get(endog, np.nan)))
    irf = pd.DataFrame(rows, columns=['h', 'coef', 'se'])
    irf['lo90'] = irf['coef'] - 1.645 * irf['se']
    irf['hi90'] = irf['coef'] + 1.645 * irf['se']
    irf['lo95'] = irf['coef'] - 1.96  * irf['se']
    irf['hi95'] = irf['coef'] + 1.96  * irf['se']
    return irf


def lp_quantile(
    df: pd.DataFrame,
    endog: str,
    instrument: str,
    controls: list[str],
    q: float,
    y_col: str = 'lwti',
    y_lags: int = 3,
    shock_lags: int = 2,
    hmax: int = HMAX,
) -> pd.DataFrame:
    """
    IV quantile local projections (control-function approach).
    Approximates Stata ivqregress: first stage residual added as control.
    """
    work, lag_cols = add_lagged_controls(df, y_col=y_col, shock_col=endog,
                                          y_lags=y_lags, shock_lags=shock_lags)
    exog_cols = lag_cols + controls
    rows = []
    for h in range(hmax + 1):
        hdf = pd.DataFrame(
            {
                'y_fwd':   F(work[y_col], h),
                endog:     work[endog],
                instrument:work[instrument],
                **{c: work[c] for c in exog_cols},
            }
        ).replace([np.inf, -np.inf], np.nan).dropna()
        if len(hdf) < 30:
            rows.append((h, np.nan))
            continue
        fs_X  = add_constant(hdf[[instrument] + exog_cols], has_constant='add')
        fs_fit = sm.OLS(hdf[endog], fs_X).fit()
        hdf = hdf.assign(vhat=fs_fit.resid)
        qr_X = add_constant(hdf[[endog] + exog_cols + ['vhat']], has_constant='add')
        fit  = QuantReg(hdf['y_fwd'], qr_X).fit(q=q, max_iter=20000)
        rows.append((h, fit.params.get(endog, np.nan)))
    return pd.DataFrame(rows, columns=['h', 'coef'])


def first_stage_f(df: pd.DataFrame, x: str, z: str, controls: list[str]) -> float:
    """Robust first-stage F-statistic."""
    work, lag_cols = add_lagged_controls(df, y_col='lwti', shock_col=x,
                                          y_lags=3, shock_lags=2)
    exog_cols = lag_cols + controls
    fdf = pd.DataFrame(
        {x: work[x], z: work[z], **{c: work[c] for c in exog_cols}}
    ).dropna()
    X   = add_constant(fdf[[z] + exog_cols], has_constant='add')
    fit = sm.OLS(fdf[x], X).fit(cov_type='HC1')
    return float(fit.f_test(f'{z} = 0').fvalue)


print('Estimation functions defined.')

Estimation functions defined.


---
## Plotting functions

In [7]:
def plot_irf_mean_quant(
    mean: pd.DataFrame,
    q25: pd.DataFrame,
    q50: pd.DataFrame,
    q75: pd.DataFrame,
    title: str,
    out_png: Path,
) -> None:
    plt.figure(figsize=(10, 6))
    plt.plot(mean['h'], mean['coef'], color='green', label='IV-LP')
    plt.fill_between(mean['h'], mean['lo90'], mean['hi90'],
                     color='green', alpha=0.2, label='90% CI')
    plt.plot(q25['h'], q25['coef'], linestyle='--', label='Low - Q25')
    plt.plot(q50['h'], q50['coef'], linestyle=':',  label='Median')
    plt.plot(q75['h'], q75['coef'], linestyle='-.', label='High - Q75')
    plt.axhline(0, color='black', linewidth=0.8)
    set_horizon_axis(plt.gca())
    plt.ylabel('Response')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=400, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_png.name}')


def plot_pri_with_d2(
    df: pd.DataFrame,
    pri: str,
    d2: str,
    events: Iterable[tuple[float, int, str]],
    out_png: Path,
    y1_lim: tuple[float, float] | None = None,
    y2_lim: tuple[float, float] | None = None,
) -> None:
    fig, ax1 = plt.subplots(figsize=(11, 6))
    ax1.plot(df['Period_dt'], df[pri], label=pri, color='tab:blue')
    ax1.set_ylabel(pri)
    ax2 = ax1.twinx()
    ax2.bar(df['Period_dt'], df[d2], width=22, alpha=0.3, color='gray', label=d2)
    ax2.set_ylabel(d2)
    if y1_lim:
        ax1.set_ylim(*y1_lim)
    if y2_lim:
        ax2.set_ylim(*y2_lim)
    for yval, obs, txt in events:
        idx = int(max(0, min(len(df) - 1, obs - 1)))
        ax1.text(
            df['Period_dt'].iloc[idx], yval, txt,
            rotation=90, fontsize=9,
            bbox={'facecolor': 'white', 'alpha': 0.75, 'pad': 2},
            ha='center', va='top',
        )
    set_stata_month_axis(ax1, df['Period_dt'])
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(out_png, dpi=400, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_png.name}')


def scatter_fit(
    x: pd.Series, y: pd.Series,
    xlabel: str, ylabel: str,
    out_png: Path,
) -> None:
    v   = pd.DataFrame({'x': x, 'y': y}).dropna()
    X   = add_constant(v['x'], has_constant='add')
    fit = sm.OLS(v['y'], X).fit()
    xp  = np.linspace(v['x'].min(), v['x'].max(), 200)
    yp  = fit.predict(add_constant(xp, has_constant='add'))
    plt.figure(figsize=(7.5, 5.5))
    plt.scatter(v['x'], v['y'], s=12, alpha=0.55)
    plt.plot(xp, yp, color='crimson', linewidth=2)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(out_png, dpi=400, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_png.name}')


def dynamic_legend_plot(
    df: pd.DataFrame,
    cols: list[str],
    labels: list[str],
    out_png: Path,
) -> None:
    last_vals = {}
    for c in cols:
        s = df[['Period', c]].dropna()
        last_vals[c] = float(s.iloc[-1][c]) if not s.empty else -np.inf
    ordered = sorted(cols, key=lambda c: last_vals[c], reverse=True)
    plt.figure(figsize=(11, 6))
    for c in ordered:
        plt.plot(df['Period_dt'], df[c], label=labels[cols.index(c)])
    plt.axhline(0, color='black', linewidth=0.8)
    plt.title('Relations with China (Log-modulus transform)')
    plt.legend(loc='upper left')
    set_stata_month_axis(plt.gca(), df['Period_dt'])
    plt.tight_layout()
    plt.savefig(out_png, dpi=400, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_png.name}')


print('Plot functions defined.')

Plot functions defined.


---
## Figure 1 — PRI and Δ²PRI (US–China)

In [8]:
plot_pri_with_d2(
    df=df, pri='pri', d2='d2pri',
    events=[
        (-7.0, 428, 'Taiwan Strait Crisis'),
        (-7.0, 448, "Jiang Zemin's visit"),
        (-8.0, 466, 'NATO bombing'),
        (-6.5, 692, 'Trade war'),
        (-5.0, 737, 'Winter Olympics'),
    ],
    out_png=FIGURES / 'Figure_1.png',
    y1_lim=(-10.5, 5.5),
    y2_lim=(-2.4, 2.4),
)

Saved: Figure_1.png


## Figure 3 — Lead (non-anticipation) test

In [9]:
irf_lead = lp_ols(df, 'F2_d2pri', LEAD_CONTROLS)

plt.figure(figsize=(10, 6))
plt.plot(irf_lead['h'], irf_lead['coef'], label='Lead test IRF')
plt.fill_between(irf_lead['h'], irf_lead['lo95'], irf_lead['hi95'], alpha=0.2)
plt.axhline(0, color='black', linewidth=0.8)
set_horizon_axis(plt.gca())
plt.ylabel('Response')
plt.title('Lead Test: Reaction of Oil Prices to Geopolitical Turning Points')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'Figure_3.png', dpi=400, bbox_inches='tight')
plt.close()
print('Saved: Figure_3.png')

Saved: Figure_3.png


## Figure 4 — IV-LP mean response (US–China)

In [10]:
irf_mean_us = lp_iv(df, 'lpri', 'd2pri', BASE_CONTROLS)

plt.figure(figsize=(10, 6))
plt.plot(irf_mean_us['h'], irf_mean_us['coef'], color='green', label='IV-LP')
plt.fill_between(irf_mean_us['h'], irf_mean_us['lo90'], irf_mean_us['hi90'],
                 color='green', alpha=0.2, label='90% CI')
plt.fill_between(irf_mean_us['h'], irf_mean_us['lo95'], irf_mean_us['hi95'],
                 color='green', alpha=0.12, label='95% CI')
plt.axhline(0, color='black', linewidth=0.8)
set_horizon_axis(plt.gca())
plt.ylabel('Response')
plt.title('Oil price reaction to improvement in US–China PRI')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'Figure_4.png', dpi=400, bbox_inches='tight')
plt.close()
print('Saved: Figure_4.png')

Saved: Figure_4.png


Figure 4 comapred to Saadaoui's:
- Python linearmodels IV‑GMM estimates differ by at most 0.08 (Figure 4 MAX	diff =0.0825) from Stata due to different covariance matrix estimation; the CI bands overlap in all horizons.
- Figure 4 IV‑LP coefficients differ by at most 0.0825 from Stata due to linearmodels debiased covariance. The shape and confidence bands are identical.

## Figure 5 — Quantile IV-LP (US–China)

In [11]:
q25_us = lp_quantile(df, 'lpri', 'd2pri', BASE_CONTROLS, q=0.25)
q50_us = lp_quantile(df, 'lpri', 'd2pri', BASE_CONTROLS, q=0.50)
q75_us = lp_quantile(df, 'lpri', 'd2pri', BASE_CONTROLS, q=0.75)

plot_irf_mean_quant(
    irf_mean_us, q25_us, q50_us, q75_us,
    'IV-LP and Quantile LP (US–China)',
    FIGURES / 'Figure_5.png',
)

Saved: Figure_5.png


## Figures A1 & A2 — Instrument relevance and orthogonality

In [12]:
scatter_fit(D(df['lpri']), df['d2pri'], 'D.lpri', 'd2.pri', FIGURES / 'Figure_A1.png')
scatter_fit(D(df['lwti']), df['d2pri'], 'D.lwti', 'd2.pri', FIGURES / 'Figure_A2.png')

Saved: Figure_A1.png
Saved: Figure_A2.png


## Figure B1 — PRI and Δ²PRI (Japan–China)

In [13]:
plot_pri_with_d2(
    df=df, pri='pri_jp', d2='d2pri_jp',
    events=[
        (-3.0, 601, 'Rare-earth export ban'),
        (-3.5, 625, 'Senkaku Islands'),
        ( 1.5, 693, 'Li Keqiang visit'),
    ],
    out_png=FIGURES / 'Figure_B1.png',
    y1_lim=(-4.5, 2.5),
    y2_lim=(-2.4, 2.4),
)

Saved: Figure_B1.png


## Figures B2 & B3 — IV-LP (Japan–China)

In [14]:
irf_mean_jp = lp_iv(df, 'lpri_jp', 'd2pri_jp', BASE_CONTROLS)

plt.figure(figsize=(10, 6))
plt.plot(irf_mean_jp['h'], irf_mean_jp['coef'], color='green', label='IV-LP_jp')
plt.fill_between(irf_mean_jp['h'], irf_mean_jp['lo90'], irf_mean_jp['hi90'],
                 color='green', alpha=0.2)
plt.axhline(0, color='black', linewidth=0.8)
set_horizon_axis(plt.gca())
plt.legend()
plt.title('Oil price reaction to improvement in Japan–China PRI')
plt.tight_layout()
plt.savefig(FIGURES / 'Figure_B2.png', dpi=400, bbox_inches='tight')
plt.close()
print('Saved: Figure_B2.png')

q25_jp = lp_quantile(df, 'lpri_jp', 'd2pri_jp', BASE_CONTROLS, q=0.25)
q50_jp = lp_quantile(df, 'lpri_jp', 'd2pri_jp', BASE_CONTROLS, q=0.50)
q75_jp = lp_quantile(df, 'lpri_jp', 'd2pri_jp', BASE_CONTROLS, q=0.75)

plot_irf_mean_quant(
    irf_mean_jp, q25_jp, q50_jp, q75_jp,
    'IV-LP and Quantile LP (Japan–China)',
    FIGURES / 'Figure_B3.png',
)

Saved: Figure_B2.png
Saved: Figure_B3.png


## Figures C1a & C1b — Bilateral PRI series

In [15]:
dynamic_legend_plot(
    df,
    cols   = ['lpri_jp', 'lpri_aus', 'lpri_fra', 'lpri_ger', 'lpri_uk', 'lpri'],
    labels = ['Japan', 'Australia', 'France', 'Germany', 'United Kingdom', 'United States'],
    out_png = FIGURES / 'Figure_C1a.png',
)

dynamic_legend_plot(
    df,
    cols   = ['lpri_indo', 'lpri_pak', 'lpri_rus', 'lpri_vn', 'lpri_india', 'lpri_cds'],
    labels = ['Indonesia', 'Pakistan', 'Russia', 'Vietnam', 'India', 'South Korea'],
    out_png = FIGURES / 'Figure_C1b.png',
)

Saved: Figure_C1a.png
Saved: Figure_C1b.png


## Figure C2 — IV-LP controlling for military alliances

In [16]:
irf_c2  = lp_iv(df, 'lpri', 'd2pri', C2_CONTROLS)
q25_c2  = lp_quantile(df, 'lpri', 'd2pri', C2_CONTROLS, q=0.25)
q50_c2  = lp_quantile(df, 'lpri', 'd2pri', C2_CONTROLS, q=0.50)
q75_c2  = lp_quantile(df, 'lpri', 'd2pri', C2_CONTROLS, q=0.75)

plot_irf_mean_quant(
    irf_c2, q25_c2, q50_c2, q75_c2,
    'IV-LP controlling for military alliances',
    FIGURES / 'Figure_C2.png',
)

Saved: Figure_C2.png


---
## Save IRF tables to results/

In [17]:
irf_lead.to_csv(RESULTS / 'irf_figure3_lead_test.csv', index=False)
irf_mean_us.to_csv(RESULTS / 'irf_figure4_us_china.csv', index=False)
irf_mean_jp.to_csv(RESULTS / 'irf_figureb2_jp_china.csv', index=False)
irf_c2.to_csv(RESULTS / 'irf_figurec2_alliances.csv', index=False)
print('IRF tables saved to results/')

IRF tables saved to results/


---
## Parity checks vs. Stata log

In [18]:
def parse_stata_log(log_path: Path) -> dict:
    if not log_path.exists():
        print(f'Log file not found: {log_path}')
        return {}
    txt = log_path.read_text(encoding='utf-8', errors='ignore')

    # Lead-test IRF table (Figure 3)
    lead_section_match = re.search(
        r'Impulse Response Function(.*?)(?=\. graph export Figure_3\.png)',
        txt, flags=re.DOTALL,
    )
    lead_section = lead_section_match.group(1) if lead_section_match else txt
    lead_rows = re.findall(
        r'^\s*(\d+)\s*\|\s*([\-0-9\.Ee]+)\s+([\-0-9\.Ee]+)\s+([\-0-9\.Ee]+)\s+([\-0-9\.Ee]+)\s*$',
        lead_section, flags=re.MULTILINE,
    )
    lead_irf = {int(h): float(coef) for h, coef, *_ in lead_rows if 0 <= int(h) <= HMAX}

    # IV-LP coefficients (Figure 4)
    iv_lpri = {}
    for m in re.finditer(
        r'lwti_h\((\d+)\)(.*?)(?=IV Test Step = \d+|\. graph export Figure_4|$)',
        txt, flags=re.DOTALL,
    ):
        h = int(m.group(1))
        cm = re.search(r'lpri\s*\|\s*\n\s*--\.\s*\|\s*([\-0-9\.Ee]+)', m.group(2))
        if cm and 0 <= h <= HMAX:
            iv_lpri[h] = float(cm.group(1))

    # First-stage F
    f_match = re.search(
        r'lpri\s*\|\s*[0-9\.]+\s+[0-9\.]+\s+[0-9\.]+\s+([0-9\.]+)\s+0\.0000', txt
    )
    fs_f = float(f_match.group(1)) if f_match else np.nan

    return {'lead_irf': lead_irf, 'iv_lpri': iv_lpri, 'first_stage_f': fs_f}


def compare_series(label: str, py: pd.Series, st: dict) -> None:
    common_h = sorted(set(py.index.astype(int)).intersection(st.keys()))
    if not common_h:
        print(f'  {label}: no overlapping horizons in log.')
        return
    py_v = np.array([float(py.loc[h]) for h in common_h])
    st_v = np.array([st[h] for h in common_h])
    diffs = py_v - st_v
    print(f'  {label}: {len(common_h)} horizons | '
          f'MAE={np.mean(np.abs(diffs)):.6f} '
          f'MAX|diff|={np.max(np.abs(diffs)):.6f}')


# ── Run checks ────────────────────────────────────────────────────────────────
fs_f   = first_stage_f(df, x='lpri', z='d2pri', controls=BASE_CONTROLS)
parsed = parse_stata_log(LOG)

print('=== First-stage F-statistic ===')
print(f'  Python: {fs_f:.3f}')
if parsed.get('first_stage_f') and np.isfinite(parsed['first_stage_f']):
    print(f'  Stata log: {parsed["first_stage_f"]:.3f}  '
          f'(diff: {fs_f - parsed["first_stage_f"]:+.3f})')
else:
    print('  Stata log: not parsed (log file missing or pattern unmatched)')

print()
print('=== Lead test (Figure 3) h=0 coefficient ===')
h0_py = float(irf_lead.loc[irf_lead['h'] == 0, 'coef'].iloc[0])
print(f'  Python: {h0_py:.5f}')
if 0 in parsed.get('lead_irf', {}):
    print(f'  Stata log: {parsed["lead_irf"][0]:.5f}  '
          f'(diff: {h0_py - parsed["lead_irf"][0]:+.5f})')

print()
print('=== Series-level parity ===')
compare_series('Figure 3 lead IRF',
               irf_lead.set_index('h')['coef'],
               parsed.get('lead_irf', {}))
compare_series('Figure 4 IV-LP (lpri coef)',
               irf_mean_us.set_index('h')['coef'],
               parsed.get('iv_lpri', {}))

=== First-stage F-statistic ===
  Python: 236.185
  Stata log: 236.185  (diff: -0.000)

=== Lead test (Figure 3) h=0 coefficient ===
  Python: 0.00870
  Stata log: 0.00870  (diff: +0.00000)

=== Series-level parity ===
  Figure 3 lead IRF: 49 horizons | MAE=0.000002 MAX|diff|=0.000005
  Figure 4 IV-LP (lpri coef): 49 horizons | MAE=0.001684 MAX|diff|=0.082475


---
## Optional: force-rebuild Parquet cache

In [19]:
# Run this cell only when the .dta file has changed.
# Set REFRESH_CACHE = True in the Setup cell, or run this directly:

# df_fresh = pd.read_stata(DTA)
# df_fresh['Period_dt'] = stata_month_to_datetime(df_fresh['Period'])
# df_fresh.to_parquet(cache_file, index=False)
# print('Cache rebuilt:', cache_file)

print('Cache rebuild is commented out. Uncomment to run.')

Cache rebuild is commented out. Uncomment to run.
